# Proyecto Productivo IB · Semana 9
## Consulta y preparación para el análisis · Equipo AgroData IC (Grupo 03)  
**Proyecto:** Sistema de alerta temprana basado en grados-día para la arañita roja (*Oligonychus punicae*) en palto Hass · Fundo Loma Verde S.A.C. (Supe, Barranca)

**Propósito:** construir un dataset analítico pertinente, trazable y reproducible a partir de la base organizada en la Semana 8 (`aranita_roja_bd_v1.sqlite`).  
**Producto:** `aranita_roja_analitico_v1.csv` + consulta SQL reproducible + validaciones + ficha de preparación.

> Hoy preparamos el insumo. El análisis descriptivo comienza en la Semana 10.

## Ruta del laboratorio  
**Base organizada → inspección → unidad de análisis → consulta SQL → JOIN → filtros → variables derivadas → validación → exportación → ficha de preparación**

**Regla central:** 1 fila = 1 evaluación fitosanitaria validada.

In [ ]:
import os, sqlite3, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)
SALIDA = Path('salidas_semana9')
SALIDA.mkdir(exist_ok=True)
print('Entorno listo. Carpeta de salidas:', SALIDA.resolve())

## 1. Cargar la base de la Semana 8
Sube `aranita_roja_bd_v1.sqlite`. Es la base validada y organizada en la Semana 8 (5 tablas: `dim_modulo`, `dim_turno`, `dim_campana`, `clima_diario`, `evaluacion_fitosanitaria`).

In [ ]:
ARCHIVO_BD = 'aranita_roja_bd_v1.sqlite'
if not os.path.exists(ARCHIVO_BD):
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('No se cargó ningún archivo.')
        nombre = list(uploaded.keys())[0]
        if nombre != ARCHIVO_BD:                  # Colab puede renombrar a 'archivo (1).sqlite'
            os.rename(nombre, ARCHIVO_BD)
    except ImportError:
        raise FileNotFoundError(f'Coloca {ARCHIVO_BD} junto al notebook y vuelve a ejecutar.')
print('Base encontrada:', ARCHIVO_BD)

## 2. Inspeccionar la base
Antes de hacer JOIN verificamos tablas, estructura y cantidad de registros.

In [ ]:
con = sqlite3.connect(ARCHIVO_BD)
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;", con)
display(tablas)
conteos = {}
for t in tablas['name']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS filas FROM {t};', con).iloc[0, 0]
    conteos[t] = int(n)
    print(f'{t:26s} -> {n} filas')
assert conteos['clima_diario'] == 183 and conteos['evaluacion_fitosanitaria'] == 1494, 'No es la base v1 de la Semana 8'

In [ ]:
for t in ['clima_diario', 'evaluacion_fitosanitaria']:
    print('\nTABLA:', t)
    display(pd.read_sql_query(f'PRAGMA table_info({t});', con)[['name', 'type', 'pk']])

## 3. Decisiones antes del código
- **Unidad de análisis:** 1 fila = 1 **evaluación fitosanitaria** (una combinación de fecha + módulo + válvula + turno), unida al clima de su día.
- **Granularidad:** detalle individual por válvula (no se agrega todavía).
- **Incluir:** `estado_calidad_evaluacion = 'ACEPTADO'` y fechas entre 2026-03-01 y 2026-08-31 (periodo del proyecto).
- **Excluir:** las 32 evaluaciones `OBSERVADO` (30 duplicados lógicos ya corregidos en la Semana 6 y 2 valores atípicos de grado de postura). **No se borran de la base**: solo no entran a este dataset.
- **Variables que no entran a la v1:** `grados_dia_oc` y `grados_dia_acumulado_ano_oc` (son el grado-día genérico del **cultivo** calculado por la estación, no el de la plaga), y las variables de clima que la pregunta actual no usa (evapotranspiración, radiación, viento, precipitación). Se pueden agregar en una v2 si el análisis lo pide.

In [ ]:
display(pd.read_sql_query('SELECT estado_calidad_evaluacion AS estado, COUNT(*) AS registros FROM evaluacion_fitosanitaria GROUP BY 1;', con))
display(pd.read_sql_query('SELECT MIN(fecha) AS fecha_min, MAX(fecha) AS fecha_max, COUNT(*) AS total FROM evaluacion_fitosanitaria;', con))

## 4. Consulta SQL integrada
Recuperamos atributos legibles desde las dimensiones y el clima del día, **sin cambiar la unidad de análisis**.

In [ ]:
SQL = '''
SELECT
  e.evaluacion_id,
  e.fecha,
  m.modulo,
  t.turno,
  e.valvula,
  e.plantas_evaluadas,
  e.aranita_hojas_infestadas     AS hojas_inf_pct,
  e.aranita_infestacion_postura  AS postura_pct,
  c.temperatura_oc,
  c.humedad_relativa,
  c.estado_calidad_clima
FROM evaluacion_fitosanitaria e
JOIN clima_diario c ON c.fecha     = e.fecha
JOIN dim_modulo   m ON m.modulo_id = e.modulo_id
JOIN dim_turno    t ON t.turno_id  = e.turno_id
WHERE e.estado_calidad_evaluacion = 'ACEPTADO'
  AND e.fecha BETWEEN '2026-03-01' AND '2026-08-31'
ORDER BY e.fecha, m.modulo, e.valvula, t.turno;
'''
print(SQL)

## 5. Control del JOIN
No basta con que el código ejecute: debemos comprobar que el JOIN **no multiplicó ni perdió filas**. El conteo esperado se calcula solo sobre la tabla de evaluaciones, sin JOIN.

In [ ]:
esperadas = pd.read_sql_query('''
SELECT COUNT(*) AS n
FROM evaluacion_fitosanitaria
WHERE estado_calidad_evaluacion = 'ACEPTADO'
  AND fecha BETWEEN '2026-03-01' AND '2026-08-31';
''', con).iloc[0, 0]
print('Filas esperadas:', esperadas)

df = pd.read_sql_query(SQL, con, parse_dates=['fecha'])
print('Dimensión:', df.shape)
display(df.head(10))

In [ ]:
assert len(df) == esperadas, 'El JOIN cambió la cantidad esperada de filas.'
assert df['evaluacion_id'].is_unique, 'evaluacion_id dejó de ser único.'
assert not df.duplicated(['fecha', 'modulo', 'valvula', 'turno']).any(), 'Hay evaluaciones repetidas (misma fecha, módulo, válvula y turno).'
print('✓ JOIN correcto. Se conserva la unidad de análisis (1,462 evaluaciones).')

## 6. Variables derivadas
Creamos los campos que necesitan las Semanas 10 a 12, **sin interpretarlos todavía**. El grado-día de la plaga (GDA) se calcula con el modelo del proyecto:

**GDA del día = máx(0, T − Tbase)**, con Tbase = 10.34 °C. El **GDA acumulado** suma los días desde el 1 de marzo de 2026 (inicio del registro de clima). Las **generaciones acumuladas** dividen ese acumulado entre K = 163.93 GDA.

> **Supuesto documentado:** Tbase y K provienen de la literatura y **no están validados en campo**. Además no hay una fecha de inicio biológica (biofix) confirmada, por eso el acumulado parte del primer día con clima.

Como el acumulado depende de **todos** los días (no solo de los días con evaluación), se calcula sobre la serie completa de clima (183 días) y después se une a cada evaluación por fecha.

In [ ]:
TBASE = 10.34       # °C · supuesto de literatura, pendiente de validación en campo
K     = 163.93      # GDA por generación · supuesto de literatura, pendiente de validación

def gda_diario(t, tbase=TBASE):
    '''GDA de un día: máx(0, T - Tbase).'''
    return max(0.0, float(t) - tbase)

# Pruebas rápidas de la fórmula (incluye el caso límite T = Tbase)
assert gda_diario(TBASE) == 0.0            # límite: no acumula
assert gda_diario(5.0) == 0.0              # por debajo de Tbase: no acumula, nunca negativo
assert abs(gda_diario(20.34) - 10.0) < 1e-9

clima = pd.read_sql_query('SELECT fecha, temperatura_oc, humedad_relativa FROM clima_diario ORDER BY fecha;', con, parse_dates=['fecha'])
assert clima['temperatura_oc'].notna().all() and clima['humedad_relativa'].notna().all()
assert clima['fecha'].is_unique and len(clima) == 183

clima['gda_dia']  = (clima['temperatura_oc'] - TBASE).clip(lower=0)
clima['gda_acum'] = clima['gda_dia'].cumsum()
clima['generaciones_acum'] = clima['gda_acum'] / K
clima['temp_7d'] = clima['temperatura_oc'].rolling(7).mean()      # promedio de los 7 días que terminan en la fecha
clima['hr_7d']   = clima['humedad_relativa'].rolling(7).mean()
display(clima.tail(3).round({c: 2 for c in clima.columns if c != 'fecha'}))
print('GDA acumulado al 30-ago:', round(clima['gda_acum'].iloc[-1], 2), '| generaciones:', round(clima['generaciones_acum'].iloc[-1], 2))

In [ ]:
derivadas = ['gda_dia', 'gda_acum', 'generaciones_acum', 'temp_7d', 'hr_7d']
n_antes = len(df)
df = df.merge(clima[['fecha'] + derivadas], on='fecha', how='left', validate='m:1')
df['mes'] = df['fecha'].dt.to_period('M').astype(str)

# Orden final de columnas y redondeo
orden = ['evaluacion_id', 'fecha', 'mes', 'modulo', 'turno', 'valvula', 'plantas_evaluadas', 'hojas_inf_pct', 'postura_pct',
         'temperatura_oc', 'humedad_relativa', 'estado_calidad_clima'] + derivadas
df = df[orden]
df[derivadas] = df[derivadas].round(2)
assert len(df) == n_antes, 'La unión con la serie de clima cambió el número de filas.'
display(df.head(8))

## 7. Validar antes de exportar
Verificamos unicidad, completitud crítica, filtro, periodo, rangos y coherencia de las variables derivadas.

In [ ]:
assert df['evaluacion_id'].is_unique
for c in ['fecha', 'modulo', 'turno', 'valvula', 'hojas_inf_pct', 'postura_pct', 'temperatura_oc', 'humedad_relativa', 'gda_dia', 'gda_acum']:
    assert df[c].notna().all(), f'Hay nulos en {c}'
assert df['hojas_inf_pct'].between(0, 100).all() and df['postura_pct'].between(0, 100).all()
assert df['humedad_relativa'].between(0, 100).all()
assert df['fecha'].min() >= pd.Timestamp('2026-03-01') and df['fecha'].max() <= pd.Timestamp('2026-08-31')
assert set(df['modulo']) == {1, 2, 3, 4, 5, 6}
assert (df['gda_dia'] >= 0).all(), 'GDA diario negativo'

# El acumulado nunca baja a medida que avanza la fecha (revisión por fecha, no por evaluación).
por_fecha = df.drop_duplicates('fecha').sort_values('fecha')
assert por_fecha['gda_acum'].is_monotonic_increasing, 'El GDA acumulado disminuye en algún punto'

# Conciliación independiente: el acumulado en la última fecha con evaluación debe ser la suma simple del clima hasta ese día.
ultima = por_fecha.iloc[-1]
suma_directa = clima.loc[clima['fecha'] <= ultima['fecha'], 'gda_dia'].sum()
assert abs(ultima['gda_acum'] - suma_directa) < 0.01
print('✓ Validaciones superadas')
print('\nTipos:'); print(df.dtypes)
print('\nDimensión final:', df.shape)

## 8. Revisión estructural
Solo comprobamos estructura y nulos. **No hacemos aún análisis descriptivo.**

In [ ]:
print('Columnas:', df.columns.tolist())
nulos = df.isna().sum().to_frame('nulos'); display(nulos[nulos['nulos'] > 0])
print('Nulos esperados: temp_7d y hr_7d no existen para los primeros 6 días del clima (la ventana de 7 días aún no se completa).')
assert df.isna().sum().drop(['temp_7d', 'hr_7d']).sum() == 0
assert df['temp_7d'].isna().sum() == df['hr_7d'].isna().sum() == 15
for c in ['modulo', 'turno', 'valvula', 'mes', 'estado_calidad_clima']:
    print(f'{c:22s}: {df[c].nunique()} valores únicos')
print('Fechas de evaluación distintas:', df['fecha'].nunique())

## 9. Exportar con versión
Se exportan tres evidencias: el dataset, la **consulta SQL** (para reproducirlo) y la ficha de preparación.

In [ ]:
SALIDA_CSV = SALIDA / 'aranita_roja_analitico_v1.csv'
df.assign(fecha=df['fecha'].dt.strftime('%Y-%m-%d')).to_csv(SALIDA_CSV, index=False, encoding='utf-8-sig')
(SALIDA / 'consulta_analitica_v1.sql').write_text(SQL.strip() + '\n', encoding='utf-8')

relectura = pd.read_csv(SALIDA_CSV, encoding='utf-8-sig')
assert relectura.shape == df.shape
print('Archivo exportado:', SALIDA_CSV)
print('Registros:', relectura.shape[0], '| Columnas:', relectura.shape[1])

## 10. Ficha de preparación

In [ ]:
ficha = pd.DataFrame({
 'Elemento': ['Proyecto', 'Versión de entrada', 'Unidad de análisis', 'Granularidad', 'Periodo', 'Regla de inclusión', 'Regla de exclusión',
              'Tablas integradas', 'Variables incluidas', 'Variables no incluidas', 'Variables derivadas', 'Parámetros del modelo',
              'Validaciones', 'Archivo de salida', 'Limitaciones conocidas'],
 'Definición': [
  'Sistema de alerta temprana basado en grados-día para la arañita roja (O. punicae) en palto Hass · Fundo Loma Verde S.A.C.',
  'aranita_roja_bd_v1.sqlite (Semana 8): 183 días de clima y 1,494 evaluaciones fitosanitarias',
  '1 fila = 1 evaluación fitosanitaria validada (fecha + módulo + válvula + turno), con el clima de su día',
  'Detalle individual por válvula (sin agregar)',
  'Evaluaciones: 2026-03-02 a 2026-08-28 (50 fechas). Clima: 2026-03-01 a 2026-08-30',
  "estado_calidad_evaluacion = 'ACEPTADO' (1,462 de 1,494 evaluaciones)",
  'Las 32 evaluaciones OBSERVADO (30 duplicados lógicos y 2 atípicos de grado de postura); permanecen en la base, no entran a este dataset',
  'evaluacion_fitosanitaria + clima_diario + dim_modulo + dim_turno',
  'evaluacion_id, fecha, modulo, turno, valvula, plantas_evaluadas, hojas_inf_pct, postura_pct, temperatura_oc, humedad_relativa, estado_calidad_clima',
  'grados_dia_oc y grados_dia_acumulado_ano_oc (grado-día del cultivo, no de la plaga); evapotranspiración, radiación, viento y precipitación; grados de severidad 0-4',
  'mes; gda_dia = máx(0, T - Tbase); gda_acum = suma de gda_dia desde 2026-03-01; generaciones_acum = gda_acum / K; temp_7d y hr_7d = promedio móvil de 7 días',
  'Tbase = 10.34 °C y K = 163.93 GDA (literatura, pendientes de validación en campo)',
  'unicidad de evaluacion_id y de (fecha, módulo, válvula, turno), completitud, rangos 0-100, periodo, conteo tras JOIN (1,462), GDA >= 0 y acumulado creciente, conciliación del acumulado',
  'aranita_roja_analitico_v1.csv (+ consulta_analitica_v1.sql)',
  'Un solo ciclo de 6 meses; sin fecha biológica de inicio (biofix); el clima viene de una estación; 3 de las 50 fechas tienen clima OBSERVADO (se conservan con su marca)']})
display(ficha)
ficha.to_csv(SALIDA / 'aranita_roja_ficha_preparacion_v1.csv', index=False, encoding='utf-8-sig')

# RETO DEL EQUIPO · Transferencia al proyecto real
Respuestas del equipo (antes de programar, como pide la docente):

1. **¿Qué representa una fila?** Una evaluación fitosanitaria de una válvula en una fecha y turno, con el clima de ese día.
2. **¿Cuál es la granularidad?** Detalle individual por válvula; la agregación por fecha, mes o módulo se hace en el análisis, no aquí.
3. **¿Qué fuentes/tablas necesitan?** `evaluacion_fitosanitaria` y `clima_diario`, más `dim_modulo` y `dim_turno` para nombres legibles.
4. **¿Qué variables son pertinentes?** % de hojas infestadas (indicador principal), % de postura, módulo, fecha, temperatura y humedad del día, y el GDA acumulado.
5. **¿Qué entra y qué se excluye?** Entran las 1,462 evaluaciones `ACEPTADO`; se excluyen las 32 `OBSERVADO` y las variables que la pregunta actual no usa.
6. **¿Mantienen detalle o agregan?** Mantienen detalle.
7. **¿Qué variables derivan?** `mes`, `gda_dia`, `gda_acum`, `generaciones_acum`, `temp_7d` y `hr_7d`.
8. **¿Cómo validarán la salida?** Con `assert`: conteo tras el JOIN, unicidad, nulos, rangos, periodo y conciliación del acumulado contra la suma directa del clima.
9. **¿Cómo demostrarán reproducibilidad?** Con este notebook, la consulta `consulta_analitica_v1.sql`, la ficha y la base `aranita_roja_bd_v1.sqlite`.

# CHECKLIST FINAL
- [x] Unidad de análisis definida
- [x] Granularidad definida
- [x] Variables justificadas
- [x] Inclusión/exclusión documentada
- [x] JOIN controlado
- [x] Variables derivadas documentadas
- [x] Validaciones superadas
- [x] Consulta/script guardado (`consulta_analitica_v1.sql`)
- [x] Salida versionada (`_v1`)
- [x] Proceso reproducible

In [ ]:
con.close()
zip_path = shutil.make_archive('semana9_evidencias_AgroDataIC', 'zip', root_dir=SALIDA)
print('Paquete creado:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Fuera de Colab, el ZIP queda guardado en la carpeta de ejecución.')
print('Conexión cerrada. Laboratorio finalizado.')